# HandGCN → TFLite FP32 logits

Chạy **Runtime → Run all**, upload ba artifact khi được hỏi. Notebook strict-load checkpoint, export, convert và kiểm tra 64 chuỗi. Output là logits; consumer Softmax đúng một lần. Không retrain/quantize.


In [ ]:
# Chỉ cài trong Colab/Linux; CPU đủ dùng.
import os, sys, subprocess
from pathlib import Path
assert sys.platform == "linux"
os.chdir("/content")
VENV = Path("/content/handgcn_converter_env")
if not (VENV / "bin/python").exists():
    subprocess.run([sys.executable, "-m", "venv", str(VENV)], check=False)
PYTHON = str(VENV / "bin/python")
if not (VENV / "bin/python").exists() or subprocess.run([PYTHON, "-m", "pip", "--version"], stdout=subprocess.DEVNULL).returncode:
    subprocess.run([sys.executable, "-m", "pip", "install", "virtualenv==20.35.3"], check=True)
    subprocess.run([sys.executable, "-m", "virtualenv", str(VENV)], check=True)
subprocess.run([PYTHON, "-m", "pip", "install", "--upgrade", "pip"], check=True)
subprocess.run([PYTHON, "-m", "pip", "install", "torch==2.11.0", "torchao==0.17.0", "--index-url", "https://download.pytorch.org/whl/cpu"], check=True)
subprocess.run([PYTHON, "-m", "pip", "install", "litert-torch==0.9.4", "litert-converter==0.4.0", "ai-edge-litert==2.2.0", "numpy==2.1.3", "torch==2.11.0", "torchao==0.17.0"], check=True)
print("Môi trường HandGCN conversion đã sẵn sàng.")


In [ ]:
from google.colab import files
from pathlib import Path
import hashlib, os
os.chdir("/content")
required = {
    "handgcn_alphabet_20260913_175720.pt": "b891cb0d157de813ce0376eceb0935dc6cc3557f31c6fd513bfdfc3449726318",
    "handgcn_alphabet_20260913_175720.json": "112c0e82c9df041af49390c399e231955ef4215877a742cb4b21e9723d31c99e",
    "deploy_manifest.json": "614076a89b1cde08b3238a8a6cdb9b3dde9169b2978f125c3340d9cc37160214",
}
if any(not Path(name).exists() for name in required):
    print("Chọn cùng lúc checkpoint, sidecar JSON và deploy_manifest.json.")
    files.upload()
for name, expected in required.items():
    path = Path(name)
    assert path.exists(), f"Thiếu {name}"
    assert hashlib.sha256(path.read_bytes()).hexdigest() == expected, f"SHA-256 sai: {name}"
print("Ba artifact đúng hash.")
TEST_DATA = Path("/content/test_data.npz")


## Converter độc lập đã nhúng đúng source `hd_gcn.py`


In [ ]:
%%writefile /content/convert_handgcn_colab.py
# Generated by scripts/alphabet/build_handgcn_colab.py.
# Direct HandGCN -> FP32 logits; no retraining, quantization or output Softmax.
from __future__ import annotations

# Original source: VOYA-Collector/processed/train_utils/models/base.py
"""Base class cho tất cả sign language models"""

from abc import ABC, abstractmethod
from typing import Any, Dict, Optional

import torch
import torch.nn as nn


def initialize_kaiming(module: nn.Module) -> None:
    """
    Initialize all Conv/Linear/RNN layers with Kaiming Normal (He et al., 2015).

    Kaiming initialization is standard for ReLU networks and ensures:
    - Consistent weight distribution across architectures
    - Proper variance scaling based on network depth
    - Fair comparison between different model types

    Reference: He et al. "Delving Deep into Rectifiers" - ICCV 2015

    Args:
        module: PyTorch module to initialize
    """
    for m in module.modules():
        if isinstance(m, (nn.Conv1d, nn.Conv2d, nn.Linear)):
            nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, (nn.LSTM, nn.GRU)):
            # For recurrent layers, initialize weights (not biases)
            for name, param in m.named_parameters():
                if 'weight_ih' in name or 'weight_hh' in name:
                    nn.init.kaiming_normal_(param, nonlinearity='relu')
                elif 'bias' in name:
                    nn.init.zeros_(param)
        elif isinstance(m, nn.BatchNorm1d):
            # BatchNorm: weights to 1, biases to 0
            if m.weight is not None:
                nn.init.ones_(m.weight)
            if m.bias is not None:
                nn.init.zeros_(m.bias)


class SignLanguageModel(ABC, nn.Module):
    """
    Abstract base class cho tất cả sign language models.

    Tất cả models phải implement:
    - forward(x) để inference
    - from_config() để tạo model từ config dict
    - get_model_name() để lấy tên model
    """

    def __init__(
        self,
        input_dim: int,
        output_dim: int,
        name: Optional[str] = None,
        **kwargs,
    ):
        """
        Args:
            input_dim: Số features đầu vào (thường 126 cho sign language)
            output_dim: Số classes đầu ra
            name: Tên model (nếu None dùng class name)
        """
        super().__init__()
        self.input_dim = input_dim
        self.output_dim = output_dim
        self._model_name = name or self.__class__.__name__

    @abstractmethod
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Forward pass

        Args:
            x: Input tensor shape (batch, seq_len, input_dim)

        Returns:
            Output logits shape (batch, output_dim)
        """
        pass

    @classmethod
    @abstractmethod
    def from_config(
        cls,
        input_dim: int,
        output_dim: int,
        config: Optional[Dict[str, Any]] = None,
    ):
        """
        Tạo model từ config dict.

        Args:
            input_dim: Input dimension
            output_dim: Output dimension
            config: Dict chứa model-specific hyperparameters
                   (ví dụ: dropout, channels, levels, kernel_size, etc.)

        Returns:
            Model instance
        """
        pass

    def get_model_name(self) -> str:
        """Lấy tên model để log"""
        return self._model_name

    def get_config(self) -> Dict[str, Any]:
        """
        Trả về config của model.
        Override trong subclass nếu cần lưu hyperparameters.
        """
        return {
            "model": self.get_model_name(),
            "input_dim": self.input_dim,
            "output_dim": self.output_dim,
        }

    def count_parameters(self) -> int:
        """Đếm tổng số parameters"""
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

    def __repr__(self) -> str:
        """Nice representation"""
        param_count = self.count_parameters()
        return f"{self.get_model_name()}(input_dim={self.input_dim}, output_dim={self.output_dim}, params={param_count:,})"


# Original source: VOYA-Collector/processed/train_utils/models/hd_gcn.py
"""Hierarchical Distance-aware Graph Convolutional Network for sign language recognition"""

from typing import Any, Dict, Optional, List

import torch
import torch.nn as nn
import torch.nn.functional as F



def normalize_adjacency(adj: torch.Tensor) -> torch.Tensor:
    """Symmetric normalization: D^(-1/2) * A * D^(-1/2)"""
    degree = adj.sum(dim=1)
    degree_inv_sqrt = torch.pow(degree + 1e-6, -0.5)
    degree_inv_sqrt[torch.isinf(degree_inv_sqrt)] = 0.0
    degree_mat_inv_sqrt = torch.diag(degree_inv_sqrt)
    return degree_mat_inv_sqrt @ adj @ degree_mat_inv_sqrt


class DistanceAwareGCNLayer(nn.Module):
    """
    Distance-aware Graph Convolutional Layer.
    Uses multiple adjacency matrices for different hop distances.
    """
    def __init__(self, in_channels: int, out_channels: int, max_distance: int = 2):
        super().__init__()
        self.max_distance = max_distance
        self.out_channels = out_channels
        # We need a linear projection for each distance matrix
        self.linear_layers = nn.ModuleList([
            nn.Linear(in_channels, out_channels) for _ in range(max_distance + 1)
        ])
        self.bn = nn.BatchNorm1d(out_channels)
        self.relu = nn.ReLU(inplace=True)

        for linear in self.linear_layers:
            nn.init.kaiming_normal_(linear.weight, nonlinearity='relu')
            if linear.bias is not None:
                nn.init.zeros_(linear.bias)

    def forward(self, x: torch.Tensor, adjs: List[torch.Tensor]) -> torch.Tensor:
        """
        Args:
            x: [B, N, in_channels]
            adjs: List of normalized adjacency matrices [N, N], length = max_distance + 1
        """
        B, N, C = x.shape

        # Gộp các phép chiếu theo khoảng cách thành MỘT gemm rồi mới tách.
        # Trọng số vẫn nằm ở từng nn.Linear (state_dict không đổi), chỉ ghép lại
        # lúc forward — rẻ vì ma trận trọng số bé, trong khi x là [batch*T, N, C].
        weight = torch.cat([lin.weight for lin in self.linear_layers], dim=0)
        bias = torch.cat([lin.bias for lin in self.linear_layers], dim=0)
        projections = F.linear(x, weight, bias).split(self.out_channels, dim=-1)

        out: Optional[torch.Tensor] = None
        for d in range(self.max_distance + 1):
            # A_d @ x_proj. matmul broadcasts [N,N] over the batch, nên không
            # phải expand adjacency thành B bản rồi bmm như trước.
            contrib = torch.matmul(adjs[d], projections[d])
            out = contrib if out is None else out + contrib

        # BatchNorm1d thống kê theo channel: gộp [B,N,C] -> [B*N,C] cho ra đúng
        # cùng mean/var với đường [B,C,N], nhưng bỏ được 2 lần permute trên
        # tensor lớn (B ở đây là batch*seq_len nên mỗi permute là một bản copy).
        out = self.bn(out.reshape(B * N, -1)).reshape(B, N, -1)
        return self.relu(out)


class HandGCNModel(SignLanguageModel):
    """
    HandGCN — a hierarchical distance-aware Graph Convolutional Network for hands.
    
    Architecture:
    - Treats hand keypoints as graph nodes.
    - Graph convolution uses distance-aware matrices (D=0, 1, 2) to capture local and semi-local dependencies.
    - Hierarchical Pooling aggregates 21 joints into 6 functional parts (Wrist + 5 Fingers).
    - Temporal convolutions and Attention model the time axis.
    """

    def __init__(
        self,
        input_dim: int,
        output_dim: int,
        num_nodes: int = 42,
        gcn_channels: int = 64,
        num_gcn_layers: int = 2,
        temporal_channels: int = 128,
        dropout: float = 0.3,
        max_distance: int = 2,
        **kwargs,
    ):
        super().__init__(input_dim, output_dim, name="HandGCN")
        self.num_nodes = num_nodes
        self.gcn_channels = gcn_channels
        self.num_gcn_layers = num_gcn_layers
        self.temporal_channels = temporal_channels
        self.dropout_rate = dropout
        self.max_distance = max_distance
        
        # Build multi-distance adjacencies for Fine level (42 nodes)
        fine_adjs = self._build_distance_adjacencies_fine(max_distance)
        for d in range(max_distance + 1):
            self.register_buffer(f"fine_adj_d{d}", fine_adjs[d])
            
        # Build multi-distance adjacencies for Part level (12 nodes: 6 parts x 2 hands)
        part_adjs = self._build_distance_adjacencies_part(max_distance)
        for d in range(max_distance + 1):
            self.register_buffer(f"part_adj_d{d}", part_adjs[d])

        # Define hierarchical mapping matrix (42 -> 12)
        self.register_buffer("hierarchical_pooling_matrix", self._build_pooling_matrix())

        self.input_projection = nn.Linear(input_dim // self.num_nodes, gcn_channels)

        # Fine-level GCNs
        self.fine_gcns = nn.ModuleList([
            DistanceAwareGCNLayer(gcn_channels if i > 0 else gcn_channels, gcn_channels, max_distance)
            for i in range(num_gcn_layers)
        ])
        
        # Part-level GCNs (Hierarchical Layer)
        self.part_gcns = nn.ModuleList([
            DistanceAwareGCNLayer(gcn_channels, gcn_channels, max_distance)
            for _ in range(1)
        ])

        # Temporal processing
        self.temporal_conv1 = nn.Conv1d(gcn_channels, temporal_channels, kernel_size=3, padding=1)
        self.temporal_conv2 = nn.Conv1d(temporal_channels, temporal_channels, kernel_size=3, padding=1)

        self.attention = nn.Sequential(
            nn.Linear(temporal_channels, temporal_channels // 2),
            nn.ReLU(inplace=True),
            nn.Linear(temporal_channels // 2, 1),
            nn.Softmax(dim=1),
        )

        self.classifier = nn.Sequential(
            nn.Linear(temporal_channels, temporal_channels // 2),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(temporal_channels // 2, output_dim),
        )

        initialize_kaiming(self)

    def _get_shortest_paths(self, num_nodes: int, edges: List[tuple]) -> torch.Tensor:
        # Floyd-Warshall to find shortest paths
        dist = torch.full((num_nodes, num_nodes), float('inf'))
        for i in range(num_nodes):
            dist[i, i] = 0
        for i, j in edges:
            dist[i, j] = 1
            dist[j, i] = 1
            
        for k in range(num_nodes):
            for i in range(num_nodes):
                for j in range(num_nodes):
                    if dist[i, j] > dist[i, k] + dist[k, j]:
                        dist[i, j] = dist[i, k] + dist[k, j]
        return dist

    def _build_distance_adjacencies_fine(self, max_distance: int) -> List[torch.Tensor]:
        edges = [
            (0, 1), (1, 2), (2, 3), (3, 4),          # Thumb
            (0, 5), (5, 6), (6, 7), (7, 8),          # Index
            (0, 9), (9, 10), (10, 11), (11, 12),     # Middle
            (0, 13), (13, 14), (14, 15), (15, 16),   # Ring
            (0, 17), (17, 18), (18, 19), (19, 20),   # Pinky
        ]
        all_edges = []
        for i, j in edges:
            all_edges.append((i, j))
            all_edges.append((i + 21, j + 21))
            
        dist_mat = self._get_shortest_paths(42, all_edges)
        
        adjs = []
        for d in range(max_distance + 1):
            adj = (dist_mat == d).float()
            adjs.append(normalize_adjacency(adj))
            
        return adjs

    def _build_distance_adjacencies_part(self, max_distance: int) -> List[torch.Tensor]:
        # 12 nodes total (6 parts per hand: wrist + 5 fingers)
        edges = [
            (0, 1), (0, 2), (0, 3), (0, 4), (0, 5)
        ]
        all_edges = []
        for i, j in edges:
            all_edges.append((i, j))
            all_edges.append((i + 6, j + 6))
            
        dist_mat = self._get_shortest_paths(12, all_edges)
        
        adjs = []
        for d in range(max_distance + 1):
            adj = (dist_mat == d).float()
            adjs.append(normalize_adjacency(adj))
            
        return adjs

    def _build_pooling_matrix(self) -> torch.Tensor:
        """
        Creates a pooling matrix P [12, 42] mapping joints to parts.
        """
        P = torch.zeros(12, 42)
        
        # Left hand (0-20) -> parts 0-5
        P[0, 0] = 1.0 # wrist
        P[1, 1:5] = 1.0 / 4 # thumb
        P[2, 5:9] = 1.0 / 4 # index
        P[3, 9:13] = 1.0 / 4 # middle
        P[4, 13:17] = 1.0 / 4 # ring
        P[5, 17:21] = 1.0 / 4 # pinky
        
        # Right hand (21-41) -> parts 6-11
        P[6, 21] = 1.0 # wrist
        P[7, 22:26] = 1.0 / 4 # thumb
        P[8, 26:30] = 1.0 / 4 # index
        P[9, 30:34] = 1.0 / 4 # middle
        P[10, 34:38] = 1.0 / 4 # ring
        P[11, 38:42] = 1.0 / 4 # pinky
        
        return P

    def encode(self, x_btd: torch.Tensor) -> torch.Tensor:
        if x_btd.ndim != 3:
            raise RuntimeError(f"Expected 3D tensor [B,T,D], got {x_btd.shape}")

        B, T, D = x_btd.shape
        features_per_node = D // self.num_nodes
        x_graph = x_btd.reshape(B * T, self.num_nodes, features_per_node)

        # 1. Input Projection
        x_graph = self.input_projection(x_graph)
        
        # 2. Fine level Distance-aware GCN
        fine_adjs = [getattr(self, f"fine_adj_d{d}") for d in range(self.max_distance + 1)]
        for gcn_layer in self.fine_gcns:
            x_graph = gcn_layer(x_graph, fine_adjs)
            
        # 3. Hierarchical Pooling (42 nodes -> 12 nodes)
        x_part = torch.matmul(self.hierarchical_pooling_matrix, x_graph)
        
        # 4. Part level Distance-aware GCN
        part_adjs = [getattr(self, f"part_adj_d{d}") for d in range(self.max_distance + 1)]
        for gcn_layer in self.part_gcns:
            x_part = gcn_layer(x_part, part_adjs)

        # 5. Global Node Pooling (Mean over 12 part nodes)
        x_temporal = x_part.mean(dim=1)
        x_temporal = x_temporal.reshape(B, T, self.gcn_channels)

        # 6. Temporal Convolution
        x_temporal = x_temporal.permute(0, 2, 1)
        x_temporal = torch.relu(self.temporal_conv1(x_temporal))
        x_temporal = torch.relu(self.temporal_conv2(x_temporal))
        
        # 7. Attention over Time
        x_attn = x_temporal.permute(0, 2, 1)
        attn_weights = self.attention(x_attn)
        x_pooled = (x_attn * attn_weights).sum(dim=1)

        return x_pooled

    def forward(self, x_btd: torch.Tensor) -> torch.Tensor:
        return self.classifier(self.encode(x_btd))

    @classmethod
    def from_config(
        cls,
        input_dim: int,
        output_dim: int,
        config: Optional[Dict[str, Any]] = None,
    ) -> "HandGCNModel":
        if config is None:
            config = {}

        return cls(
            input_dim=input_dim,
            output_dim=output_dim,
            num_nodes=config.get("num_nodes", 42),
            gcn_channels=config.get("gcn_channels", 64),
            num_gcn_layers=config.get("num_gcn_layers", 2),
            temporal_channels=config.get("temporal_channels", 128),
            dropout=config.get("dropout", 0.3),
            max_distance=config.get("max_distance", 2),
        )

    def get_config(self) -> Dict[str, Any]:
        return {
            "model": "HandGCN",
            "input_dim": self.input_dim,
            "output_dim": self.output_dim,
            "num_nodes": self.num_nodes,
            "gcn_channels": self.gcn_channels,
            "num_gcn_layers": self.num_gcn_layers,
            "temporal_channels": self.temporal_channels,
            "dropout": self.dropout_rate,
            "max_distance": self.max_distance,
        }


# Original source: VOYA-Collector/processed/shared/normalization.py
import numpy as np

def normalize_single_hand(hand: np.ndarray) -> np.ndarray:
    """
    Normalize ONE hand independently.

    hand shape: (21,3)
    """

    h = hand.astype(np.float32).copy()

    # empty hand
    if not np.any(h):
        return h

    # wrist landmark
    wrist = h[0, :2].copy()

    # translate
    h[:, :2] = h[:, :2] - wrist

    # compute scale
    valid = np.linalg.norm(h[:, :2], axis=1) > 1e-6

    if valid.any():

        pts = h[valid, :2]

        span_x = pts[:,0].max() - pts[:,0].min()
        span_y = pts[:,1].max() - pts[:,1].min()

        scale = max(span_x, span_y)

        if scale > 1e-6:
            h[:, :2] = h[:, :2] / scale

    return h

def normalize_hands_vector_126(vec: np.ndarray) -> np.ndarray:

    if vec is None:
        return vec

    v = np.asarray(vec, dtype=np.float32)

    if v.size != 126:
        return v

    try:
        arr = v.reshape(2, 21, 3).astype(np.float32)
    except Exception:
        return v

    # preserve semantic hand identity
    left = arr[0]
    right = arr[1]

    # normalize independently
    left = normalize_single_hand(left)
    right = normalize_single_hand(right)

    out = np.concatenate([
        left.reshape(-1),
        right.reshape(-1)
    ]).astype(np.float32)

    return out

EXPECTED_FILE_SHA256 = {'handgcn_alphabet_20260913_175720.pt': 'b891cb0d157de813ce0376eceb0935dc6cc3557f31c6fd513bfdfc3449726318', 'handgcn_alphabet_20260913_175720.json': '112c0e82c9df041af49390c399e231955ef4215877a742cb4b21e9723d31c99e', 'deploy_manifest.json': '614076a89b1cde08b3238a8a6cdb9b3dde9169b2978f125c3340d9cc37160214'}
MODEL_SOURCE_SHA256 = {'VOYA-Collector/processed/train_utils/models/base.py': '1cd3150297c54d41caa84d12545fd67dd7b4312f7a38ca64110a3d8143df1484', 'VOYA-Collector/processed/train_utils/models/hd_gcn.py': '50bb4faba9f4723206dca8f8e9db167a9bb7909b75dcaa33ac7aae2c08d2317a', 'VOYA-Collector/processed/shared/normalization.py': '2c416100a9aca58be3cf5f0bf31f6559dd69ba089eaf12355dee8311828028bc'}
EMBEDDED_SOURCE_COMMIT = 'e3afc3ec18763e382528b3e1bc2caa78a3829cec'

import argparse
import hashlib
import importlib.metadata
import json
import platform
import sys
from pathlib import Path

STEM = "handgcn_alphabet_20260913_175720"
ATOL = 1e-4
RTOL = 1e-4


def sha(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


def write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False) + "\n", encoding="utf-8")


def versions():
    result = {"python": sys.version, "platform": platform.platform()}
    for package in ("torch", "numpy", "litert-torch", "litert-converter", "torchao", "ai-edge-litert"):
        try:
            result[package] = importlib.metadata.version(package)
        except importlib.metadata.PackageNotFoundError:
            result[package] = None
    return result


def indexed(mapping, index):
    return mapping[index] if index in mapping else mapping[str(index)]


def load_checkpoint(path, metadata_path=None, manifest_path=None):
    path = Path(path).resolve()
    if sha(path) != EXPECTED_FILE_SHA256[STEM + ".pt"]:
        raise ValueError("Checkpoint SHA-256 differs from the reviewed HandGCN checkpoint")
    checkpoint = torch.load(path, map_location="cpu", weights_only=True)
    required = {
        "model_state_dict", "model_type", "model_config", "feature_dim", "seq_len", "num_classes",
        "idx_to_label", "label_to_idx", "normalization_version", "preprocess_contract",
    }
    if required - checkpoint.keys():
        raise ValueError(f"Checkpoint is missing metadata: {sorted(required - checkpoint.keys())}")
    if str(checkpoint["model_type"]).lower().replace("-", "").replace("_", "") not in {"handgcn", "hdgcn"}:
        raise ValueError(f"Expected HandGCN, got {checkpoint['model_type']!r}")
    if (checkpoint["seq_len"], checkpoint["feature_dim"], checkpoint["num_classes"]) != (60, 126, 30):
        raise ValueError("HandGCN checkpoint must be [60,126] with 30 classes")
    if checkpoint["normalization_version"] != "hands126_v1":
        raise ValueError("Unsupported normalization contract")
    contract = checkpoint["preprocess_contract"]
    if contract.get("expects_strict_shape") != [60, 126] or contract.get("coordinate_order") != "xyz":
        raise ValueError("Checkpoint preprocessing shape/order mismatch")

    config = dict(checkpoint["model_config"])
    if config.pop("model", "HandGCN") != "HandGCN":
        raise ValueError("model_config is not HandGCN")
    input_dim = int(config.pop("input_dim"))
    output_dim = int(config.pop("output_dim"))
    allowed = {"num_nodes", "gcn_channels", "num_gcn_layers", "temporal_channels", "dropout", "max_distance"}
    if set(config) != allowed:
        raise ValueError(f"Unexpected HandGCN config keys: {sorted(set(config) ^ allowed)}")
    model = HandGCNModel(input_dim=input_dim, output_dim=output_dim, **config).cpu().eval()
    model.load_state_dict(checkpoint["model_state_dict"], strict=True)
    if model.get_config() != checkpoint["model_config"]:
        raise ValueError("Resolved architecture differs from checkpoint model_config")

    rich = {i: indexed(checkpoint["idx_to_label"], i) for i in range(30)}
    labels = {str(i): str(rich[i]["label_original"]) for i in range(30)}
    if len(set(labels.values())) != 30 or len(checkpoint["label_to_idx"]) != 30:
        raise ValueError("Invalid label map")
    for i, entry in rich.items():
        if int(checkpoint["label_to_idx"][entry["label_key"]]) != i:
            raise ValueError(f"label_to_idx mismatch at class {i}")

    sidecar = None
    if metadata_path:
        metadata_path = Path(metadata_path).resolve()
        if sha(metadata_path) != EXPECTED_FILE_SHA256[STEM + ".json"]:
            raise ValueError("Training sidecar SHA-256 mismatch")
        sidecar = json.loads(metadata_path.read_text(encoding="utf-8"))
        if sidecar.get("config", {}).get("train_csv") != checkpoint.get("training_config", {}).get("train_csv"):
            raise ValueError("Training sidecar belongs to another run")
    manifest = None
    if manifest_path:
        manifest_path = Path(manifest_path).resolve()
        if sha(manifest_path) != EXPECTED_FILE_SHA256["deploy_manifest.json"]:
            raise ValueError("Deploy manifest SHA-256 mismatch")
        manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
        if manifest.get("checkpoint", {}).get("sha256") != sha(path):
            raise ValueError("Deploy manifest points to another checkpoint")
        if manifest.get("model_config") != checkpoint["model_config"]:
            raise ValueError("Deploy manifest architecture mismatch")
        if [x["label_original"] for x in manifest["output"]["labels"]] != [labels[str(i)] for i in range(30)]:
            raise ValueError("Deploy manifest label order mismatch")
        if manifest.get("output", {}).get("activation") != "softmax":
            raise ValueError("Deploy manifest must request serving softmax")
    return checkpoint, model, labels, sidecar, manifest


def make_fixtures():
    raw = np.zeros((64, 60, 126), dtype=np.float32)
    names = []
    for sample in range(64):
        hands = raw[sample].reshape(60, 2, 21, 3)
        mode = sample % 4
        names.append(("left", "right", "both", "empty")[mode] + f"_{sample}")
        for hand in range(2):
            if mode == 3 or (mode == 0 and hand == 1) or (mode == 1 and hand == 0):
                continue
            phase = np.linspace(0, 2 * np.pi, 60, dtype=np.float32)[:, None]
            joint = np.arange(21, dtype=np.float32)[None, :]
            wrist_x = np.float32(0.25 + 0.45 * hand + 0.002 * sample)
            wrist_y = np.float32(0.55 - 0.001 * sample)
            hands[:, hand, :, 0] = wrist_x + (joint / 20 - .5) * (.18 + .03 * np.sin(phase + sample * .07))
            hands[:, hand, :, 1] = wrist_y + (joint % 5 / 4 - .5) * (.25 + .02 * np.cos(phase * 1.2))
            hands[:, hand, :, 2] = -.12 * joint / 20 + .015 * np.sin(phase + joint)
            hands[:, hand, 0, 0] = wrist_x
            hands[:, hand, 0, 1] = wrist_y
            hands[:, hand, 0, 2] = 0
        if sample % 7 == 0:
            hands[20:24] = 0
        if sample % 9 == 0:
            hands[35, :, 8:11] = 0
    normalized = np.stack([
        np.stack([normalize_hands_vector_126(frame) for frame in clip]) for clip in raw
    ]).astype(np.float32)
    if not np.isfinite(normalized).all():
        raise ValueError("Non-finite structured fixture")
    return names, raw, normalized


def read_test_data(path):
    if not path:
        return [], None, None
    data = np.load(path, allow_pickle=False)
    x = np.asarray(data["x"], dtype=np.float32)
    y = np.asarray(data["y"], dtype=np.int64)
    if x.ndim != 3 or x.shape[1:] != (60, 126) or y.shape != (len(x),) or not np.isfinite(x).all():
        raise ValueError("test_data.npz needs finite x [N,60,126] and y [N]")
    if np.any((y < 0) | (y >= 30)):
        raise ValueError("Test labels must be 0..29")
    ids = [str(v) for v in data["sample_ids"]] if "sample_ids" in data else [f"test_{i}" for i in range(len(x))]
    return ids, x, y


def predict_torch(model, x):
    values = []
    with torch.inference_mode():
        for clip in x:
            values.append(model(torch.from_numpy(clip[None])).cpu().numpy())
    return np.concatenate(values)


def compare(expected, actual, names):
    a = np.asarray(expected, np.float64)
    b = np.asarray(actual, np.float64)
    if a.shape != b.shape or not np.isfinite(a).all() or not np.isfinite(b).all():
        raise ValueError(f"Invalid parity tensors: {a.shape} vs {b.shape}")
    error = np.abs(a - b)
    relative = error / np.maximum(np.abs(a), 1e-6)
    changed = np.flatnonzero(a.argmax(-1) != b.argmax(-1))
    return {
        "sampleCount": len(a), "argmaxAgreement": float(1 - len(changed) / len(a)),
        "maxAbsoluteError": float(error.max()), "meanAbsoluteError": float(error.mean()),
        "maxRelativeError": float(relative.max()), "meanRelativeError": float(relative.mean()),
        "atol": ATOL, "rtol": RTOL, "allclose": bool(np.allclose(a, b, atol=ATOL, rtol=RTOL)),
        "changedArgmax": [{"sample": names[i], "pytorch": int(a[i].argmax()), "other": int(b[i].argmax())} for i in changed],
    }


def inspect_tflite(path):
    from ai_edge_litert.interpreter import Interpreter
    runtime = Interpreter(model_path=str(path), num_threads=4)
    runtime.allocate_tensors()
    inputs = runtime.get_input_details()
    outputs = runtime.get_output_details()
    if len(inputs) != 1 or len(outputs) != 1:
        raise ValueError("Expected exactly one TFLite input/output")
    tin, tout = inputs[0], outputs[0]
    for tensor, shape in ((tin, [1, 60, 126]), (tout, [1, 30])):
        if tensor["shape"].tolist() != shape or tensor["dtype"] != np.float32:
            raise ValueError("TFLite shape/dtype mismatch")
        if tensor["quantization"] != (0.0, 0) or len(tensor["quantization_parameters"]["scales"]):
            raise ValueError("Quantized tensor found; FP32 conversion required")
    signatures = runtime.get_signature_list()
    if len(signatures) > 1:
        raise ValueError("Expected zero or one signature")
    if signatures:
        signature_key, spec = next(iter(signatures.items()))
        if len(spec["inputs"]) != 1 or len(spec["outputs"]) != 1:
            raise ValueError("Signature must contain one input/output")
        input_key, output_key = spec["inputs"][0], spec["outputs"][0]
        runner = runtime.get_signature_runner(signature_key)
        predict = lambda x: runner(**{input_key: x})[output_key]
    else:
        signature_key, input_key, output_key = "", tin["name"], tout["name"]
        def predict(x):
            runtime.set_tensor(tin["index"], x)
            runtime.invoke()
            return runtime.get_tensor(tout["index"])
    metadata = {
        "signatures": signatures, "signatureKey": signature_key,
        "frameInputName": input_key, "outputName": output_key,
        "input": {"name": tin["name"], "shape": tin["shape"].tolist(), "dtype": str(tin["dtype"]), "quantization": list(tin["quantization"])},
        "output": {"name": tout["name"], "shape": tout["shape"].tolist(), "dtype": str(tout["dtype"]), "quantization": list(tout["quantization"])},
    }
    return runtime, metadata, predict


def edge_predict(edge, x):
    value = edge(x)
    if isinstance(value, (tuple, list)):
        if len(value) != 1:
            raise ValueError("Converted model returned multiple outputs")
        value = value[0]
    return np.asarray(value, dtype=np.float32)


def main():
    parser = argparse.ArgumentParser(description="Strict HandGCN checkpoint audit and FP32 LiteRT conversion")
    parser.add_argument("command", choices=("audit", "convert"))
    parser.add_argument("--checkpoint", default=STEM + ".pt")
    parser.add_argument("--metadata")
    parser.add_argument("--deploy-manifest")
    parser.add_argument("--test-data")
    parser.add_argument("--output-dir", required=True)
    args = parser.parse_args()
    out = Path(args.output_dir).resolve()
    if out.exists():
        raise FileExistsError(f"Refusing to reuse output directory: {out}")
    out.mkdir(parents=True)

    checkpoint, model, labels, sidecar, manifest = load_checkpoint(args.checkpoint, args.metadata, args.deploy_manifest)
    names, raw, x = make_fixtures()
    test_names, test_x, y = read_test_data(args.test_data)
    if test_x is not None:
        names += test_names
        x = np.concatenate([x, test_x])
    expected = predict_torch(model, x)
    sample = torch.from_numpy(x[:1])
    exported_program = torch.export.export(model, (sample,), strict=True)
    exported = []
    with torch.inference_mode():
        for clip in x:
            exported.append(exported_program.module()(torch.from_numpy(clip[None])).numpy())
    exported = np.concatenate(exported)
    report = {
        "versions": versions(), "checkpointSha256": sha(args.checkpoint),
        "checkpointModelType": checkpoint["model_type"], "checkpointGitCommit": checkpoint.get("git_commit"),
        "embeddedSourceCommit": EMBEDDED_SOURCE_COMMIT, "modelSourceSha256": MODEL_SOURCE_SHA256,
        "resolvedModelConfig": model.get_config(), "strictLoad": True,
        "normalizationVersion": checkpoint["normalization_version"], "preprocessContract": checkpoint["preprocess_contract"],
        "runPurpose": checkpoint.get("run_purpose"), "checkpointMetrics": checkpoint.get("metrics"),
        "sidecarVerified": sidecar is not None, "deployManifestVerified": manifest is not None,
        "fixtureCount": 64, "testDataCount": 0 if test_x is None else len(test_x),
        "torchExportParity": compare(expected, exported, names),
        "outputContract": "FP32 logits; Android/consumer applies stable softmax exactly once",
        "newConversionExecuted": False, "afterConvertParity": None, "serializedTfliteParity": None,
    }
    write_json(out / "report.json", report)
    write_json(out / "labels.json", labels)
    checkpoint_metadata = {k: v for k, v in checkpoint.items() if k != "model_state_dict"}
    write_json(out / "checkpoint.json", checkpoint_metadata)
    np.savez_compressed(out / "fixtures.npz", raw=raw, x=x[:64], pytorch_logits=expected[:64], sample_ids=np.asarray(names[:64]))
    raw.astype("<f4").tofile(out / "raw.f32")
    x[:64].astype("<f4").tofile(out / "input.f32")
    if args.command == "audit":
        report["auditPassed"] = report["torchExportParity"]["allclose"] and report["torchExportParity"]["argmaxAgreement"] == 1
        write_json(out / "report.json", report)
        print(json.dumps({"auditPassed": report["auditPassed"], "torchExportParity": report["torchExportParity"]}, indent=2))
        return

    try:
        import litert_torch
    except ImportError as exc:
        report["conversionBlocker"] = "litert_torch is unavailable; run conversion on Linux/Colab"
        write_json(out / "report.json", report)
        raise RuntimeError(report["conversionBlocker"]) from exc
    print("Converting HandGCN directly to FP32 logits...", flush=True)
    # LiteRT-Torch 0.9.4 requires sample_args to be a tuple of tensors.
    edge = litert_torch.convert(model.cpu().eval(), (sample,))
    converted = np.concatenate([edge_predict(edge, clip[None]) for clip in x])
    report["afterConvertParity"] = compare(expected, converted, names)
    binary = out / (STEM + "_verified_logits_fp32.tflite")
    edge.export(str(binary))
    report["newConversionExecuted"] = True
    _, metadata, predict = inspect_tflite(binary)
    actual = np.concatenate([predict(clip[None]) for clip in x])
    report["serializedTfliteParity"] = compare(expected, actual, names)
    report["convertVsReloadParity"] = compare(converted, actual, names)
    report["tfliteSha256"] = sha(binary)
    report["tfliteMetadata"] = metadata
    # A direct conversion must retain logits. A hidden Softmax would fail numerical parity above.
    checks = [report["torchExportParity"], report["afterConvertParity"], report["serializedTfliteParity"], report["convertVsReloadParity"]]
    report["pythonParityPassed"] = all(x["allclose"] and x["argmaxAgreement"] == 1 for x in checks)
    if test_x is not None:
        offset = 64
        report["accuracy"] = {
            "n": len(y), "pytorch": float((expected[offset:].argmax(-1) == y).mean()),
            "tflite": float((actual[offset:].argmax(-1) == y).mean()),
        }
    else:
        report["accuracy"] = None
    labels_name = STEM + "_verified_labels.json"
    write_json(out / labels_name, labels)
    write_json(out / "registry-entry.json", {
        "displayName": "HandGCN Alphabet FP32 logits (verified)", "modelFile": binary.name,
        "labelsFile": labels_name, "sequenceLength": 60, "featureDimension": 126, "classCount": 30,
        "signatureKey": metadata["signatureKey"], "frameInputName": metadata["frameInputName"],
        "lengthInputName": "", "outputName": metadata["outputName"],
        "inputTensorName": metadata["input"]["name"], "outputTensorName": metadata["output"]["name"],
        "applySoftmax": True, "mirrorInput": False, "swapHandedness": True,
        "normalizationVersion": "alphabet_hands126_v1", "sampleFps": 30, "numThreads": 4,
        "modelSha256": sha(binary), "labelsSha256": sha(out / labels_name), "outputType": "logits",
        "sourceCheckpointSha256": report["checkpointSha256"],
    })
    write_json(out / "golden.json", {
        "modelSha256": sha(binary), "checkpointSha256": report["checkpointSha256"], "sampleNames": names[:64],
        "shape": [64, 60, 126], "labels": labels, "pytorchLogits": expected[:64].tolist(),
        "tfliteLogits": actual[:64].tolist(), "rawSha256": sha(out / "raw.f32"),
        "inputSha256": sha(out / "input.f32"), "atol": ATOL, "rtol": RTOL,
    })
    write_json(out / "report.json", report)
    print(json.dumps({"pythonParityPassed": report["pythonParityPassed"], "serializedTfliteParity": report["serializedTfliteParity"], "accuracy": report["accuracy"]}, indent=2))
    if not report["pythonParityPassed"]:
        raise RuntimeError("PARITY FAILED: do not deploy this TFLite")


if __name__ == "__main__":
    main()


In [ ]:
import datetime, json, shutil, subprocess
from pathlib import Path
from google.colab import files
RUN = Path("/content") / ("handgcn_export_" + datetime.datetime.now().strftime("%Y%m%d_%H%M%S_%f"))
LOG = Path(str(RUN) + ".log")
cmd = [PYTHON, "/content/convert_handgcn_colab.py", "convert",
       "--checkpoint", "/content/handgcn_alphabet_20260913_175720.pt", "--metadata", "/content/handgcn_alphabet_20260913_175720.json",
       "--deploy-manifest", "/content/deploy_manifest.json", "--output-dir", str(RUN)]
if TEST_DATA.exists(): cmd += ["--test-data", str(TEST_DATA)]
with LOG.open("w", encoding="utf-8") as log:
    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, encoding="utf-8", errors="replace", bufsize=1)
    for line in process.stdout:
        print(line, end=""); log.write(line)
    exit_code = process.wait()
RUN.mkdir(exist_ok=True)
shutil.copyfile(LOG, RUN / "conversion.log")
with (RUN / "requirements-frozen.txt").open("w") as f:
    subprocess.run([PYTHON, "-m", "pip", "freeze"], stdout=f, check=True)
report_path = RUN / "report.json"
report = json.loads(report_path.read_text(encoding="utf-8")) if report_path.exists() else {}
passed = exit_code == 0 and report.get("pythonParityPassed") is True and report.get("newConversionExecuted") is True
status = "PASS" if passed else "FAILED"
print(status, json.dumps(report.get("serializedTfliteParity"), indent=2))
archive = shutil.make_archive(str(RUN) + "_" + status, "zip", root_dir=RUN)
print("Gửi lại ZIP:", Path(archive).name)
files.download(archive)
